# Apple data in Ontario

In [18]:
import pandas as pd
import numpy as np

xl = pd.ExcelFile("ctyapple_en.xlsx")
sheet_names = xl.sheet_names

drop_rows = [
    "Southern District", "Western District", "Central District",
    "Eastern District", "Northern District", "Province"
]

all_years = []

for sheet in sheet_names:
    year = int(sheet.replace("ctyapple", "").strip())
    df = pd.read_excel("ctyapple_en.xlsx", sheet_name=sheet, header=3)
    
    if df.shape[1] > 5:
        df = df.iloc[:, :5]
    
    df.columns = ["county", "harvested_area_acres", "marketed_prod_000lbs",
                  "avg_price_cents_lb", "farm_value_000cad"]
    
    df = df[df["county"].notna()]
    df["county"] = df["county"].str.strip()
    
    df["county"] = df["county"].replace({
        "Prescott & Russell": "Prescott and Russell",
        "Ottawa-Carleton": "Ottawa",
    })
    
    df = df[~df["county"].isin(drop_rows)]
    df = df[~df["county"].str.contains(
        "Note|Reference|Census|OMAF", na=False, case=False)]
    
    for col in ["harvested_area_acres", "marketed_prod_000lbs",
                "avg_price_cents_lb", "farm_value_000cad"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    
    df["year"] = year
    all_years.append(df)

panel = pd.concat(all_years, ignore_index=True)
panel = panel.dropna(subset=["harvested_area_acres", "marketed_prod_000lbs"])

other = panel[panel["county"] == "Other"].groupby("year").agg({
    "harvested_area_acres": "sum",
    "marketed_prod_000lbs": "sum",
    "farm_value_000cad": "sum",
    "avg_price_cents_lb": "mean",  
    "county": "first"
}).reset_index(drop=True)
other["county"] = "Other"

panel = panel[panel["county"] != "Other"]
panel = pd.concat([panel, other], ignore_index=True)

panel["yield_lbs_per_acre"] = (
    panel["marketed_prod_000lbs"] * 1000) / panel["harvested_area_acres"]
panel["log_yield"] = np.log(panel["yield_lbs_per_acre"].replace(0, np.nan))
panel["log_price"] = np.log(panel["avg_price_cents_lb"].replace(0, np.nan))
panel["log_prod"] = np.log(panel["marketed_prod_000lbs"].replace(0, np.nan))
panel["log_area"] = np.log(panel["harvested_area_acres"].replace(0, np.nan))

print("Shape:", panel.shape)
print("Counties:", sorted(panel["county"].unique()))
print("Years:", sorted(panel["year"].unique()))

dups = panel.groupby(["county", "year"]).size().reset_index(name="count")
print("\nDuplicates:")
print(dups[dups["count"] > 1])

panel.to_csv("apple_panel_base.csv", index=False)

Shape: (1009, 11)
Counties: ['Algoma', 'Brant', 'Bruce', 'Chatham-Kent', 'Dufferin', 'Durham', 'Eastern and Northern Districts', 'Elgin', 'Essex', 'Frontenac', 'Grey', 'Haldimand-Norfolk', 'Halton', 'Hamilton', 'Hamilton-Wentworth', 'Hasting', 'Hastings', 'Huron', 'Kawartha Lakes', 'Kent', 'Lambton', 'Lanark', 'Leeds & Grenville', 'Lennox & Addington', 'Manitoulin', 'Middlesex', 'Niagara', 'Northumberland', 'Other', 'Other Districts', 'Ottawa', 'Oxford', 'Peel', 'Perth', 'Peterborough', 'Prescott and Russell', 'Prince Edward', 'Renfrew', 'Simcoe', 'Stormont, Dundas & Glengarry', 'Thunder Bay', 'Waterloo', 'Wellington', 'York']
Years: [np.float64(1995.0), np.float64(1996.0), np.float64(1997.0), np.float64(1998.0), np.float64(1999.0), np.float64(2000.0), np.float64(2001.0), np.float64(2002.0), np.float64(2003.0), np.float64(2004.0), np.float64(2005.0), np.float64(2006.0), np.float64(2007.0), np.float64(2008.0), np.float64(2009.0), np.float64(2010.0), np.float64(2011.0), np.float64(2012.0

In [19]:
other = panel[panel["county"] == "Other"].groupby("year").agg({
    "harvested_area_acres": "sum",
    "marketed_prod_000lbs": "sum",
    "farm_value_000cad": "sum",
    "avg_price_cents_lb": "mean",
}).reset_index()  
other["county"] = "Other"

panel = panel[panel["county"] != "Other"]
panel = pd.concat([panel, other], ignore_index=True)

panel["yield_lbs_per_acre"] = (
    panel["marketed_prod_000lbs"] * 1000) / panel["harvested_area_acres"]
panel["log_yield"] = np.log(panel["yield_lbs_per_acre"].replace(0, np.nan))
panel["log_price"] = np.log(panel["avg_price_cents_lb"].replace(0, np.nan))
panel["log_prod"] = np.log(panel["marketed_prod_000lbs"].replace(0, np.nan))
panel["log_area"] = np.log(panel["harvested_area_acres"].replace(0, np.nan))


panel.to_csv("apple_panel_base.csv", index=False)

In [20]:
panel.head(10)

,county,harvested_area_acres,marketed_prod_000lbs,avg_price_cents_lb,farm_value_000cad,year,yield_lbs_per_acre,log_yield,log_price,log_prod,log_area
0,Brant,203.0,4984.0,40.50,2019.0,2024.0,24551.724138,10.108537,3.701302,8.513988,5.313206
1,Chatham-Kent,242.0,6219.0,36.90,2295.0,2024.0,25698.347107,10.154182,3.608212,8.735364,5.488938
2,Elgin,1077.0,28110.0,39.70,11160.0,2024.0,26100.278552,10.169701,3.681351,10.243881,6.981935
3,Essex,1004.0,25100.0,36.20,9086.0,2024.0,25000.000000,10.126631,3.589059,10.130623,6.911747
4,Haldimand-Norfolk,1828.0,48442.0,39.60,19183.0,2024.0,26500.000000,10.184900,3.678829,10.788122,7.510978
5,Hamilton,168.0,4124.0,39.90,1645.0,2024.0,24547.619048,10.108370,3.686376,8.324579,5.123964
6,Lambton,431.0,10991.0,36.20,3979.0,2024.0,25501.160093,10.146479,3.589059,9.304832,6.066108
7,Middlesex,522.0,13415.0,38.00,5098.0,2024.0,25699.233716,10.154216,3.637586,9.504129,6.257668
8,Niagara,606.0,15514.0,38.30,5942.0,2024.0,25600.660066,10.150373,3.645450,9.649498,6.406880
9,Oxford,91.0,2339.0,38.49,900.0,2024.0,25703.296703,10.154375,3.650398,7.757479,4.510860


In [21]:
print(panel['year'].unique())
print(panel['county'].unique())

[2024. 2023. 2022. 2021. 2020. 2019. 2018. 2017. 2016. 2015. 2014. 2013.
 2012. 2011. 2010. 2009. 2008. 2007. 2006. 2005. 2004. 2003. 2002. 2001.
 2000. 1999. 1998. 1997. 1996. 1995.]
<StringArray>
[                         'Brant',                   'Chatham-Kent',
                          'Elgin',                          'Essex',
              'Haldimand-Norfolk',                       'Hamilton',
                        'Lambton',                      'Middlesex',
                        'Niagara',                         'Oxford',
                          'Bruce',                       'Dufferin',
                           'Grey',                         'Halton',
                          'Huron',                           'Peel',
                          'Perth',                         'Simcoe',
                       'Waterloo',                     'Wellington',
                         'Durham',                       'Hastings',
                 'Northumberland',         

# Annual Exchange Rates 

I downloaded the CSV file of annual exchange rate for the years before 2017 from Historical noon and closing rates, Bank of Canada. https://www.bankofcanada.ca/rates/exchange/legacy-noon-and-closing-rates/. 

The file is under section "Annual average rates". 

In [22]:
df = pd.read_csv('LEGACY_ANNUAL_RATES.csv', skiprows=94)

df['date'] = pd.to_datetime(df['date'])
df['Year'] = df['date'].dt.year

usd_rates = df[(df['Year'] >= 2010) & (df['Year'] <= 2016)][['Year', 'IEXA0101']]
usd_rates.columns = ['Year', 'USD_to_CAD_Average']

print(usd_rates)

    Year  USD_to_CAD_Average
13  2010            1.029939
14  2011            0.989069
15  2012            0.999580
16  2013            1.029915
17  2014            1.104466
18  2015            1.278711
19  2016            1.324806


I downloaded the CSV file of annual exchange rate for the years 2017 to 2024 from up-to-date website Annual exchange rates, Bank of Canada. https://www.bankofcanada.ca/rates/exchange/annual-average-exchange-rates/. 

In [23]:
df2 = pd.read_csv('FX_RATES_ANNUAL-sd-2017-01-01.csv', skiprows=39)

df2['date'] = pd.to_datetime(df2['date'])
df2['Year'] = df2['date'].dt.year

usd_rates2 = df2[(df2['Year'] >= 2017) & (df2['Year'] <= 2024)][['Year', 'FXAUSDCAD']]
usd_rates2.columns = ['Year', 'USD_to_CAD_Average']

print(usd_rates2)

   Year  USD_to_CAD_Average
0  2017              1.2986
1  2018              1.2957
2  2019              1.3269
3  2020              1.3415
4  2021              1.2535
5  2022              1.3013
6  2023              1.3497
7  2024              1.3698


In [24]:
usd_rates_10_24 = pd.concat([usd_rates, usd_rates2]).sort_values('Year')

# Apple data in Washington state

From USDA quick stats at https://quickstats.nass.usda.gov/, I got a time-series data of apple prices in Washington annualy from 2010 to 2024 by selecting: 

- Program: SURVEY
- Sector: CROPS
- Group: FRUIT & TREE NUTS
- Commodity: APPLES
- Category: PRICE RECEIVED
- Data Item: APPLES - PRICE RECEIVED, MEASURED IN $/LB
- Domain: TOTAL
- Geographic Level: STATE
- State: WASHINGTON
- Year: 2010 to 2024
- Period Type: ANNUAL
- Period: MARKETING YEAR






In [25]:
wa_prices = pd.read_csv('9B54D3E9-8CDD-3F84-B727-E8FB1F4C065C.csv')
wa_data = wa_prices[['Year', 'Value']].rename(columns={'Value': 'WA_Price_USD'})

## Price Conversion

# Merging with apple data in ontario

In [26]:
wa_data = pd.merge(wa_data, usd_rates_10_24, on='Year', how='inner')
wa_data['WA_Price_CAD'] = wa_data['WA_Price_USD'] * wa_data['USD_to_CAD_Average']
wa_data = wa_data.rename(columns={'Year': 'year'})

merged_panel = pd.merge(panel, wa_data[['year', 'WA_Price_CAD']], on='year', how='left')

merged_panel['WA_Price_cents_CAD'] = merged_panel['WA_Price_CAD'] * 100
merged_panel['log_wa_price'] = np.log(merged_panel['WA_Price_cents_CAD'])

merged_panel.to_csv('apple_panel_final.csv', index=False)
merged_panel.head(10)

,county,harvested_area_acres,marketed_prod_000lbs,avg_price_cents_lb,farm_value_000cad,year,yield_lbs_per_acre,log_yield,log_price,log_prod,log_area,WA_Price_CAD,WA_Price_cents_CAD,log_wa_price
0,Brant,203.0,4984.0,40.50,2019.0,2024.0,24551.724138,10.108537,3.701302,8.513988,5.313206,0.372586,37.25856,3.617882
1,Chatham-Kent,242.0,6219.0,36.90,2295.0,2024.0,25698.347107,10.154182,3.608212,8.735364,5.488938,0.372586,37.25856,3.617882
2,Elgin,1077.0,28110.0,39.70,11160.0,2024.0,26100.278552,10.169701,3.681351,10.243881,6.981935,0.372586,37.25856,3.617882
3,Essex,1004.0,25100.0,36.20,9086.0,2024.0,25000.000000,10.126631,3.589059,10.130623,6.911747,0.372586,37.25856,3.617882
4,Haldimand-Norfolk,1828.0,48442.0,39.60,19183.0,2024.0,26500.000000,10.184900,3.678829,10.788122,7.510978,0.372586,37.25856,3.617882
5,Hamilton,168.0,4124.0,39.90,1645.0,2024.0,24547.619048,10.108370,3.686376,8.324579,5.123964,0.372586,37.25856,3.617882
6,Lambton,431.0,10991.0,36.20,3979.0,2024.0,25501.160093,10.146479,3.589059,9.304832,6.066108,0.372586,37.25856,3.617882
7,Middlesex,522.0,13415.0,38.00,5098.0,2024.0,25699.233716,10.154216,3.637586,9.504129,6.257668,0.372586,37.25856,3.617882
8,Niagara,606.0,15514.0,38.30,5942.0,2024.0,25600.660066,10.150373,3.645450,9.649498,6.406880,0.372586,37.25856,3.617882
9,Oxford,91.0,2339.0,38.49,900.0,2024.0,25703.296703,10.154375,3.650398,7.757479,4.510860,0.372586,37.25856,3.617882


In [27]:
cols_to_keep = ['county', 'year', 'log_prod', 'log_price', 'log_wa_price']
apple_panel_washington = merged_panel[cols_to_keep]

apple_panel_washington.to_csv('apple_panel_washington.csv', index=False)

print(apple_panel_washington.head(20))

               county    year   log_prod  log_price  log_wa_price
0               Brant  2024.0   8.513988   3.701302      3.617882
1        Chatham-Kent  2024.0   8.735364   3.608212      3.617882
2               Elgin  2024.0  10.243881   3.681351      3.617882
3               Essex  2024.0  10.130623   3.589059      3.617882
4   Haldimand-Norfolk  2024.0  10.788122   3.678829      3.617882
5            Hamilton  2024.0   8.324579   3.686376      3.617882
6             Lambton  2024.0   9.304832   3.589059      3.617882
7           Middlesex  2024.0   9.504129   3.637586      3.617882
8             Niagara  2024.0   9.649498   3.645450      3.617882
9              Oxford  2024.0   7.757479   3.650398      3.617882
10              Bruce  2024.0   7.211557   3.648057      3.617882
11           Dufferin  2024.0   6.054439   3.589059      3.617882
12               Grey  2024.0  11.335162   3.586293      3.617882
13             Halton  2024.0   7.789455   3.713572      3.617882
14        

# Frost days

In [31]:
import pandas as pd
import numpy as np

county_centroids = {
    'Niagara':        (43.10, -79.20),
    'Haldimand-Norfolk': (42.85, -80.27),
    'Hamilton':       (43.25, -79.87),
    'Halton':         (43.53, -79.87),
    'Peel':           (43.70, -79.76),
    'York':           (44.00, -79.46),
    'Durham':         (44.00, -78.90),
    'Northumberland': (44.10, -77.95),
    'Prince Edward':  (43.95, -77.20),
    'Hastings':       (44.50, -77.50),
    'Lennox and Addington': (44.40, -76.90),
    'Frontenac':      (44.60, -76.65),
    'Leeds and Grenville': (44.70, -75.85),
    'Stormont, Dundas and Glengarry': (45.05, -74.85),
    'Prescott and Russell': (45.50, -74.85),
    'Ottawa':         (45.42, -75.70),
    'Lanark':         (45.00, -76.30),
    'Renfrew':        (45.50, -77.00),
    'Simcoe':         (44.45, -79.70),
    'Dufferin':       (44.10, -80.20),
    'Wellington':     (43.80, -80.40),
    'Waterloo':       (43.45, -80.50),
    'Perth':          (43.50, -81.00),
    'Oxford':         (43.10, -80.75),
    'Brant':          (43.13, -80.35),
    'Middlesex':      (43.00, -81.25),
    'Elgin':          (42.75, -81.10),
    'Chatham-Kent':   (42.40, -82.20),
    'Essex':          (42.15, -82.80),
    'Lambton':        (42.95, -82.10),
    'Huron':          (43.70, -81.50),
    'Bruce':          (44.50, -81.30),
    'Grey':           (44.40, -80.65),
    'Peterborough':   (44.55, -78.20),
    'Kawartha Lakes': (44.35, -78.75),
    'Manitoulin':     (45.75, -82.20),
    'Algoma':         (47.00, -84.00),
    'Thunder Bay':    (48.40, -89.25),
}

fd = pd.read_csv('annual_frost_days.csv')

records = []
for cty, (lat, lon) in county_centroids.items():
    fd['dist'] = np.sqrt((fd['Latitude'] - lat)**2 + (fd['Longitude'] - lon)**2)
    nearest = fd.nsmallest(5, 'dist')
    for yr in range(2010, 2021):
        records.append({
            'county': cty,
            'year': yr,
            'frost_days': nearest[str(yr)].mean()
        })

county_fd = pd.DataFrame(records)

panel = pd.read_csv('apple_panel_washington.csv')
panel['year'] = panel['year'].astype(int)
panel = panel[panel['year'].between(2010, 2020)].copy()

panel_fix = {
    'Hasting': 'Hastings',
    'Kent': 'Chatham-Kent',
    'Hamilton-Wentworth': 'Hamilton',
    'Leeds & Grenville': 'Leeds and Grenville',
    'Lennox & Addington': 'Lennox and Addington',
    'Stormont, Dundas & Glengarry': 'Stormont, Dundas and Glengarry',
}
panel['county'] = panel['county'].replace(panel_fix)
panel = panel[~panel['county'].isin(
    ['Eastern and Northern Districts', 'Other Districts', 'Other'])]

merged = pd.merge(panel, county_fd, on=['county', 'year'], how='left')
merged['log_frost'] = np.log(merged['frost_days'].replace(0, np.nan))

print("Shape:", merged.shape)
print("Missing frost_days:", merged['frost_days'].isna().sum())
print("Unmatched counties:", merged[merged['frost_days'].isna()]['county'].unique())
print(merged.head(10))

merged.to_csv('apple_panel_with_frost.csv', index=False)

Shape: (407, 7)
Missing frost_days: 0
Unmatched counties: <StringArray>
[]
Length: 0, dtype: str
              county  year   log_prod  log_price  log_wa_price  frost_days  \
0              Brant  2020   8.578288   3.465736      3.756394       127.0   
1       Chatham-Kent  2020   8.732305   3.443618      3.756394       101.2   
2              Elgin  2020  10.400741   3.481240      3.756394       104.4   
3              Essex  2020  10.181612   3.449988      3.756394        79.8   
4  Haldimand-Norfolk  2020  10.913487   3.397858      3.756394       102.0   
5           Hamilton  2020   9.019422   3.367296      3.756394       115.8   
6            Lambton  2020   9.585278   3.401197      3.756394       104.4   
7          Middlesex  2020   9.589598   3.446808      3.756394       125.2   
8            Niagara  2020   9.732106   3.459466      3.756394        76.0   
9             Oxford  2020   7.087574   3.496508      3.756394       129.4   

   log_frost  
0   4.844187  
1   4.617099  